# AC-MOT v11 candidate — not final results
A0–A2 retain reference logic; A3 tests low-confidence recovery and stable resolution. A4 remains legacy ablation.
Do not tune on test-dev. Freeze settings on train/validation before a full test-dev run.
Legacy recorder metrics are diagnostic only. Report TrackEval outputs from the last cell.


In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 1 — SETUP + 17 SEQUENCES (~20 min on Colab free T4)
# ════════════════════════════════════════════════════════════════
!pip install ultralytics motmetrics opencv-python-headless pandas numpy tqdm scipy lap pyyaml -q

import os, time, shutil, gc, yaml
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict, deque
from dataclasses import dataclass

import cv2
import numpy as np
import pandas as pd
import torch
import motmetrics as mm
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive

try:
    torch.backends.cudnn.benchmark = True
except Exception:
    pass

drive.mount('/content/drive', force_remount=False)

DATASET_ROOT  = Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
SEQ_DIR       = DATASET_ROOT / 'sequences'
ANNOT_DIR     = DATASET_ROOT / 'annotations'
DRIVE_RESULTS = Path('/content/drive/MyDrive/VisDrone_Results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
LOCAL_TMP     = Path('/content/_acmot_v10_tmp')

assert SEQ_DIR.exists() and ANNOT_DIR.exists(), 'Dataset path not found'
all_sequences = sorted([d for d in SEQ_DIR.iterdir() if d.is_dir()])

# ── 3 sequences × 3 systems = ~20 min on T4 ─────────────────────
VAL_SEQS_NAMES = [s.name for s in all_sequences]
by_name  = {s.name: s for s in all_sequences}
VAL_SEQS = [by_name[n] for n in VAL_SEQS_NAMES if n in by_name]

MODEL_NAME = 'yolov8n.pt'
DEVICE     = '0' if torch.cuda.is_available() else 'cpu'
HALF=False

print(f'AC-MOT v10 | Detector: {MODEL_NAME} | Device={DEVICE} | FP16={HALF}')
print(f'Running on {len(VAL_SEQS)} sequences (~20 min):')
for s in VAL_SEQS:
    print(f'  - {s.name}')
import sys
assert len(VAL_SEQS)==17
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)
print('GPU:',torch.cuda.get_device_name(0))

# v11 uses explicit FP32 to match recorded actual precision.


In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 2 — AC-MOT MODULES (v10 improved)
# ════════════════════════════════════════════════════════════════

@dataclass
class SceneState:
    sci: float = 0.0
    scene: str = 'clear'
    brightness: float = 128.0
    blur: float = 500.0
    edge_density: float = 0.0
    crowd: float = 0.0
    tiny_ratio: float = 0.0
    n_dets: int = 0


class SceneAnalyzer:
    """
    v10 fix: tightened crowd/SCI thresholds to stop over-classifying
    clear UAV sequences (brightness 120-200) as crowded.
    Key changes vs v9:
      - crowd > 0.65  (was 0.55)  — needs more objects before 'crowded'
      - edge_density > 0.13 (was 0.10) — less sensitive to texture
      - SCI crowd weight 0.35→0.30, edge weight 0.25→0.20,
        tiny weight 0.25→0.30 (tiny objects matter more for UAV)
    """
    def __init__(self, window: int = 7):
        self.sci_hist = deque(maxlen=window)

    def analyze(self, img: np.ndarray, prev_boxes: np.ndarray) -> SceneState:
        small = cv2.resize(img, (0, 0), fx=0.25, fy=0.25)
        gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)

        brightness  = float(gray.mean())
        blur        = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        edge_density= float(cv2.Canny(gray, 50, 120).mean() / 255.0)
        n_dets      = len(prev_boxes)
        crowd       = min(n_dets / 30.0, 1.0)          # v10: divisor 25→30

        if n_dets:
            areas      = ((prev_boxes[:, 2] - prev_boxes[:, 0]) *
                          (prev_boxes[:, 3] - prev_boxes[:, 1]))
            tiny_ratio = float(np.mean(areas < 32 * 32))
        else:
            tiny_ratio = 0.0

        # v10: rebalanced weights
        raw_sci = (0.30 * crowd
                 + 0.20 * min(edge_density / 0.14, 1.0)
                 + 0.30 * tiny_ratio)
        if brightness < 80:
            raw_sci += 0.10
        if blur < 180:
            raw_sci += 0.05

        self.sci_hist.append(float(np.clip(raw_sci, 0.0, 1.0)))
        sci = float(np.mean(self.sci_hist))

        # v10: tighter scene thresholds
        if brightness < 80:
            scene = 'night'
        elif blur < 180:
            scene = 'blur'
        elif tiny_ratio > 0.50:
            scene = 'tiny'
        elif crowd > 0.65 or edge_density > 0.13:   # v9 was 0.55 / 0.10
            scene = 'crowded'
        else:
            scene = 'clear'

        return SceneState(sci=sci, scene=scene, brightness=brightness,
                          blur=blur, edge_density=edge_density,
                          crowd=crowd, tiny_ratio=tiny_ratio, n_dets=n_dets)

    def reset(self):
        self.sci_hist.clear()


class SmartCalibrator:
    """
    v10 fix:
      - conf floor raised 0.17→0.19 (prevents FP explosion on UAV small objects)
      - conf ceiling kept 0.28 (safe for VisDrone)
      - imgsz thresholds unchanged (640/736/832 ladder)
    """
    def __init__(self, adaptive_threshold: bool = True,
                       adaptive_resolution: bool = True):
        self.adaptive_threshold  = adaptive_threshold
        self.adaptive_resolution = adaptive_resolution

    def params(self, state: SceneState) -> dict:
        conf  = 0.25
        iou   = 0.45
        imgsz = 640

        if self.adaptive_threshold:
            conf = 0.245 - 0.050 * state.sci          # v10: slope 0.055→0.050 (gentler)
            iou  = 0.490 - 0.050 * state.sci
            if state.scene in ['crowded', 'tiny', 'night']:
                conf -= 0.012                          # v10: nudge 0.015→0.012
            if state.scene == 'blur':
                iou -= 0.012

        if self.adaptive_resolution:
            if state.sci > 0.60 or state.tiny_ratio > 0.50:
                imgsz = 832
            elif state.sci > 0.35 or state.scene in ['crowded', 'tiny']:
                imgsz = 736

        return dict(
            conf  = float(np.clip(conf,  0.19, 0.28)),   # v10: floor 0.17→0.19
            iou   = float(np.clip(iou,   0.40, 0.52)),
            imgsz = int(imgsz)
        )


class LightweightReID:
    """
    Kept for ablation only — NOT used in production system in v10.
    v10 ablation proved ReID increases IDS on VisDrone (A3: 157 vs A2: 128).
    """
    def __init__(self, crop=24, bank=4, threshold=0.85, max_age=30):
        self.crop       = crop
        self.bank_size  = bank
        self.threshold  = threshold          # v10: raised 0.82→0.85 (stricter matching)
        self.max_age    = max_age            # v10: reduced 35→30 (shorter memory)
        self.bank       = defaultdict(lambda: deque(maxlen=bank))
        self.lost_feat  = {}
        self.lost_age   = {}
        self.seen_ids   = set()
        self.frame_idx  = 0

    def _feature(self, img, box):
        x1, y1, x2, y2 = [int(max(0, v)) for v in box]
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            return None
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop
        feat = cv2.resize(gray, (self.crop, self.crop)).ravel().astype(np.float32)
        feat -= feat.mean()
        norm = np.linalg.norm(feat)
        return feat / norm if norm > 1e-6 else None

    def update_and_remap(self, img, ids, boxes):
        self.frame_idx += 1
        current  = set(ids.tolist()) if len(ids) else set()
        remapped = ids.copy()

        for i, tid in enumerate(ids):
            tid  = int(tid)
            feat = self._feature(img, boxes[i])
            if feat is None:
                continue
            if tid not in self.seen_ids and self.lost_feat:
                best_tid, best_sim = tid, self.threshold
                for old_tid, old_feat in list(self.lost_feat.items()):
                    sim = float(np.dot(feat, old_feat))
                    if sim > best_sim:
                        best_tid, best_sim = old_tid, sim
                if best_tid != tid:
                    remapped[i] = best_tid
                    self.lost_feat.pop(best_tid, None)
                    self.lost_age.pop(best_tid, None)
                    tid = best_tid
            self.bank[tid].append(feat)
            self.seen_ids.add(tid)

        for tid in list(self.seen_ids):
            if tid not in current and tid not in self.lost_feat and self.bank[tid]:
                mean_feat = np.mean(np.stack(self.bank[tid]), axis=0)
                norm = np.linalg.norm(mean_feat)
                if norm > 1e-6:
                    self.lost_feat[tid] = mean_feat / norm
                    self.lost_age[tid]  = self.frame_idx

        for tid, age in list(self.lost_age.items()):
            if self.frame_idx - age > self.max_age:
                self.lost_feat.pop(tid, None)
                self.lost_age.pop(tid, None)
        return remapped


# ── Helper functions ────────────────────────────────────────────

def load_gt(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    cols = ['frame','id','x','y','w','h','score','cat','trunc','occ']
    df   = pd.read_csv(path, header=None, names=cols)
    df   = df[df['cat'].isin([1,4,5,6,9])]
    df   = df[(df['occ'] < 2) & (df['trunc'] < 2) & (df['score'] == 1)]
    return df.reset_index(drop=True)


def iou_dist(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    if not len(pred) or not len(gt):
        return np.empty((len(gt), len(pred)))
    ix1   = np.maximum(pred[:, 0:1].T, gt[:, 0:1])
    iy1   = np.maximum(pred[:, 1:2].T, gt[:, 1:2])
    ix2   = np.minimum(pred[:, 2:3].T, gt[:, 2:3])
    iy2   = np.minimum(pred[:, 3:4].T, gt[:, 3:4])
    inter = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)
    ap    = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
    ag    = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])
    union = ap[np.newaxis, :] + ag[:, np.newaxis] - inter
    return 1.0 - np.where(union > 0, inter / union, 0.0)


def hota_approx(tp: int, fp: int, fn: int, ids: int) -> float:
    det_a = tp / max(tp + fp + fn, 1)
    ass_a = max(0.0, 1.0 - ids / max(tp, 1))
    return float(np.sqrt(det_a * ass_a))


def eval_acc(acc, name='seq') -> dict:
    mh   = mm.metrics.create()
    summ = mh.compute(acc,
        metrics=['mota','idf1','num_switches','recall','precision',
                 'num_misses','num_false_positives','num_matches'],
        name=name)
    row  = summ.iloc[0]
    return dict(
        mota      = float(row['mota']),
        idf1      = float(row['idf1']),
        recall    = float(row['recall']),
        precision = float(row['precision']),
        ids       = int(row['num_switches']),
        fn        = int(row['num_misses']),
        fp        = int(row['num_false_positives']),
        matches   = int(row['num_matches']),
        hota      = hota_approx(
            int(row['num_matches']), int(row['num_false_positives']),
            int(row['num_misses']),  int(row['num_switches'])),
    )


def build_tracker_yaml(name, high, low, new, buffer, match) -> str:
    path = Path(f'/content/{name}.yaml')
    data = dict(tracker_type='bytetrack',
                track_high_thresh=float(high),
                track_low_thresh=float(low),
                new_track_thresh=float(new),
                track_buffer=int(buffer),
                match_thresh=float(match),
                fuse_score=True)
    path.write_text(yaml.safe_dump(data, sort_keys=False), encoding='utf-8')
    return str(path)


def reset_tracker(model):
    if getattr(model, 'predictor', None) is not None:
        model.predictor = None


print('AC-MOT v10 modules ready')
"""Candidate A3 controller. Values are experimental, not optimized results."""
class StableCalibrator:
    def __init__(self, adaptive_threshold=True, adaptive_resolution=True):
        self.adaptive_threshold = adaptive_threshold
        self.adaptive_resolution = adaptive_resolution
        self.current = 640
        self.pending = None
        self.count = 0
        self.last_state = None

    def params(self, state):
        target = 640
        if self.adaptive_resolution:
            if state.sci > .60 or state.tiny_ratio > .50:
                target = 832
            elif state.sci > .35 or state.scene in ['crowded', 'tiny']:
                target = 736
        # Count distinct analyzer updates, not repeated calls on the same state.
        if state is not self.last_state:
            self.count = self.count + 1 if target == self.pending else 1
            self.pending = target
            self.last_state = state
            if self.count >= 3:
                self.current = target
        # Allow ByteTrack's low-confidence recovery stage to see detections.
        # Track birth remains controlled by new_track_thresh in tracker YAML.
        conf = .04 if self.adaptive_threshold else .25
        iou = max(.40, min(.52, .490 - .050 * state.sci)) if self.adaptive_threshold else .45
        return dict(conf=conf, iou=iou, imgsz=self.current)


In [ ]:
# ════════════════════════════════════════════════════════════════
#  CELL 3 — RUNNER + SYSTEMS DEFINITION
# ════════════════════════════════════════════════════════════════

TRACKERS = {
    # Official default ByteTrack — no tuning
    'baseline': 'bytetrack.yaml',
    # Tuned ByteTrack — used by both Baseline_TunedTracker AND AC-MOT_v10
    # high=0.18 : ByteTrack 2nd-round uses lower-conf detections → better recall
    # buffer=45 : longer track memory → fewer ID resets
    # match=0.86: stricter IoU matching → fewer wrong associations
    # new=0.20  : v10 fix, reduces spurious new tracks vs v9 (was 0.18)
    'acmot': build_tracker_yaml('bytetrack_v10_acmot',
                                high=0.18, low=0.04,
                                new=0.20,  buffer=45, match=0.86),
}

# ── 3 production systems ─────────────────────────────────────────
# System 1 vs System 2 → isolates: tracker YAML tuning contribution
# System 2 vs System 3 → isolates: adaptive scene intelligence contribution
SYSTEMS = [
    dict(name='Baseline_Default',
         model=MODEL_NAME, tracker='baseline',
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),

    dict(name='Baseline_TunedTracker',
         model=MODEL_NAME, tracker='acmot',       # same tuned yaml as AC-MOT
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),        # zero adaptive logic

    dict(name='AC-MOT_v10',
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True, adaptive_resolution=True,
         scene_analysis=True, reid=False),         # ReID OFF — v9 ablation proved it increases IDS
]

# ── Ablation systems (4 sequences) ──────────────────────────────
# A0 → A1: What does tuned YAML alone give?
# A1 → A2: What does adaptive threshold add on top of tuned YAML?
# A2 → A3: What does adaptive resolution add?
# A3 → A4: What does ReID do? (kept as negative reference)
ABLATION_SYSTEMS = [
    dict(name='A0_Baseline_Default',
         model=MODEL_NAME, tracker='baseline',
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),

    dict(name='A1_TunedTracker_Only',
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=False, adaptive_resolution=False,
         scene_analysis=False, reid=False),

    dict(name='A2_TunedTracker_AdaptThresh',
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True,  adaptive_resolution=False,
         scene_analysis=True,  reid=False),

    dict(name='A3_TunedTracker_AdaptThresh_Res',   # = production config
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True,  adaptive_resolution=True,
         scene_analysis=True,  reid=False),

    dict(name='A4_Full_WithReID',                   # negative reference
         model=MODEL_NAME, tracker='acmot',
         adaptive_threshold=True,  adaptive_resolution=True,
         scene_analysis=True,  reid=True),
]


def get_params(system: dict, state: SceneState) -> dict:
    if system.get('adaptive_threshold') or system.get('adaptive_resolution'):
        return SmartCalibrator(
            adaptive_threshold  = system.get('adaptive_threshold', False),
            adaptive_resolution = system.get('adaptive_resolution', False)
        ).params(state)
    return dict(conf=0.25, iou=0.45, imgsz=640)


def run_system(system: dict, seqs, run_tag: str) -> pd.DataFrame:
    model = YOLO(system['model'])
    if HALF:
        model.model.model.half()

    rows = []
    for seq in tqdm(seqs, desc=system['name']):
        gt         = load_gt(ANNOT_DIR / f'{seq.name}.txt')
        frames_drv = sorted(seq.glob('*.jpg'))
        if gt.empty or not frames_drv:
            continue

        LOCAL_TMP.mkdir(exist_ok=True)
        local_seq = LOCAL_TMP / seq.name
        if local_seq.exists():
            shutil.rmtree(local_seq)
        shutil.copytree(seq, local_seq)
        frames = sorted(local_seq.glob('*.jpg'))

        reset_tracker(model)
        Calibrator = StableCalibrator if system.get('candidate_v11') else SmartCalibrator
        calibrator   = Calibrator(
                            system.get('adaptive_threshold', False),
                            system.get('adaptive_resolution', False))
        analyzer     = SceneAnalyzer()
        reid         = LightweightReID() if system.get('reid', False) else None
        acc          = mm.MOTAccumulator(auto_id=True)
        times        = []
        prev_boxes   = np.empty((0, 4))
        state        = SceneState()
        scene_counts = Counter()
        imgsz_log, conf_log = [], []

        for idx, fp in enumerate(frames, start=1):
            t0  = time.perf_counter()
            img = cv2.imread(str(fp))
            if img is None:
                continue

            if system.get('scene_analysis') and (idx == 1 or idx % 10 == 1):
                state = analyzer.analyze(img, prev_boxes)

            scene_counts[state.scene] += 1
            params = calibrator.params(state)
            imgsz_log.append(params['imgsz'])
            conf_log.append(params['conf'])

            res = model.track(
                source  = img,
                tracker = TRACKERS[system['tracker']],
                conf    = params['conf'],
                iou     = params['iou'],
                imgsz   = params['imgsz'],
                # quantize omitted
                persist = True,
                verbose = False,
                device  = DEVICE,
            )
            times.append(time.perf_counter() - t0)

            if res[0].boxes.id is not None:
                pred_ids   = res[0].boxes.id.cpu().numpy().astype(int)
                pred_boxes = res[0].boxes.xyxy.cpu().numpy()
            else:
                pred_ids   = np.array([], dtype=int)
                pred_boxes = np.empty((0, 4))
            prev_boxes = pred_boxes.copy()

            if reid is not None and len(pred_ids):
                pred_ids = reid.update_and_remap(img, pred_ids, pred_boxes)

            gt_f     = gt[gt['frame'] == idx]
            gt_ids   = gt_f['id'].values
            gt_boxes = (np.column_stack([
                gt_f['x'].values, gt_f['y'].values,
                gt_f['x'].values + gt_f['w'].values,
                gt_f['y'].values + gt_f['h'].values,
            ]) if len(gt_f) else np.empty((0, 4)))

            dist = iou_dist(pred_boxes, gt_boxes)
            acc.update(gt_ids, pred_ids,
                       dist if dist.size else np.empty((len(gt_ids), len(pred_ids))))

        shutil.rmtree(local_seq, ignore_errors=True)
        metrics = eval_acc(acc, seq.name)
        fps     = 1.0 / np.mean(times) if times else 0.0
        dom     = scene_counts.most_common(1)[0][0] if scene_counts else 'unknown'

        rows.append(dict(
            run_tag        = run_tag,
            system         = system['name'],
            tracker_cfg    = system['tracker'],       # logged for reproducibility
            sequence       = seq.name,
            frames         = len(frames),
            fps            = round(fps, 2),
            dominant_scene = dom,
            mean_imgsz     = round(float(np.mean(imgsz_log)), 1) if imgsz_log else 640,
            mean_conf      = round(float(np.mean(conf_log)),  4) if conf_log  else 0.25,
            **metrics,
        ))
        tqdm.write(
            f"{system['name']:<34} {seq.name[:24]:24s} "
            f"MOTA={metrics['mota']:.3f} IDF1={metrics['idf1']:.3f} "
            f"HOTA={metrics['hota']:.3f} IDS={metrics['ids']:4d} "
            f"FPS={fps:.1f} scene={dom}"
        )

    if HALF:
        torch.cuda.empty_cache()
    gc.collect()
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for system, g in df.groupby('system', sort=False):
        rows.append(dict(
            system     = system,
            sequences  = len(g),
            mota       = g['mota'].mean(),
            idf1       = g['idf1'].mean(),
            recall     = g['recall'].mean(),
            precision  = g['precision'].mean(),
            hota       = g['hota'].mean(),
            ids        = int(g['ids'].sum()),
            fn         = int(g['fn'].sum()),
            fp         = int(g['fp'].sum()),
            matches    = int(g['matches'].sum()),
            fps        = g['fps'].mean(),
            mean_imgsz = g['mean_imgsz'].mean(),
        ))
    out = pd.DataFrame(rows)
    if len(out) >= 2:
        base = out.iloc[0]
        for col in ['mota','idf1','recall','precision','hota','fps']:
            out[col + '_delta'] = out[col] - float(base[col])
        out['ids_delta'] = out['ids'] - int(base['ids'])
        out['fn_delta']  = out['fn']  - int(base['fn'])
        out['fp_delta']  = out['fp']  - int(base['fp'])
    return out


print('Runner ready')
print('Systems: Baseline_Default | Baseline_TunedTracker | AC-MOT_v10')
print('Tracker configs logged per-row for reproducibility')
ABLATION_SYSTEMS[3]['candidate_v11'] = True
ABLATION_SYSTEMS[3]['name'] = 'A3_v11_Candidate'


In [ ]:
code_cells = [{'source': ['# ════════════════════════════════════════════════════════════════\n', '#  CELL 1 — SETUP + 17 SEQUENCES (~20 min on Colab free T4)\n', '# ════════════════════════════════════════════════════════════════\n', '!pip install ultralytics motmetrics opencv-python-headless pandas numpy tqdm scipy lap pyyaml -q\n', '\n', 'import os, time, shutil, gc, yaml\n', 'from pathlib import Path\n', 'from datetime import datetime\n', 'from collections import Counter, defaultdict, deque\n', 'from dataclasses import dataclass\n', '\n', 'import cv2\n', 'import numpy as np\n', 'import pandas as pd\n', 'import torch\n', 'import motmetrics as mm\n', 'from tqdm import tqdm\n', 'from ultralytics import YOLO\n', 'from google.colab import drive\n', '\n', 'try:\n', '    torch.backends.cudnn.benchmark = True\n', 'except Exception:\n', '    pass\n', '\n', "drive.mount('/content/drive', force_remount=False)\n", '\n', "DATASET_ROOT  = Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')\n", "SEQ_DIR       = DATASET_ROOT / 'sequences'\n", "ANNOT_DIR     = DATASET_ROOT / 'annotations'\n", "DRIVE_RESULTS = Path('/content/drive/MyDrive/VisDrone_Results')\n", 'DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)\n', "LOCAL_TMP     = Path('/content/_acmot_v10_tmp')\n", '\n', "assert SEQ_DIR.exists() and ANNOT_DIR.exists(), 'Dataset path not found'\n", 'all_sequences = sorted([d for d in SEQ_DIR.iterdir() if d.is_dir()])\n', '\n', '# ── 3 sequences × 3 systems = ~20 min on T4 ─────────────────────\n', 'VAL_SEQS_NAMES = [s.name for s in all_sequences]\n', 'by_name  = {s.name: s for s in all_sequences}\n', 'VAL_SEQS = [by_name[n] for n in VAL_SEQS_NAMES if n in by_name]\n', '\n', "MODEL_NAME = 'yolov8n.pt'\n", "DEVICE     = '0' if torch.cuda.is_available() else 'cpu'\n", 'HALF=False\n', '\n', "print(f'AC-MOT v10 | Detector: {MODEL_NAME} | Device={DEVICE} | FP16={HALF}')\n", "print(f'Running on {len(VAL_SEQS)} sequences (~20 min):')\n", 'for s in VAL_SEQS:\n', "    print(f'  - {s.name}')\n", 'import sys\n', 'assert len(VAL_SEQS)==17\n', "assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)\n", "print('GPU:',torch.cuda.get_device_name(0))\n", '\n', '# v11 uses explicit FP32 to match recorded actual precision.\n']}, {'source': ['# ════════════════════════════════════════════════════════════════\n', '#  CELL 2 — AC-MOT MODULES (v10 improved)\n', '# ════════════════════════════════════════════════════════════════\n', '\n', '@dataclass\n', 'class SceneState:\n', '    sci: float = 0.0\n', "    scene: str = 'clear'\n", '    brightness: float = 128.0\n', '    blur: float = 500.0\n', '    edge_density: float = 0.0\n', '    crowd: float = 0.0\n', '    tiny_ratio: float = 0.0\n', '    n_dets: int = 0\n', '\n', '\n', 'class SceneAnalyzer:\n', '    """\n', '    v10 fix: tightened crowd/SCI thresholds to stop over-classifying\n', '    clear UAV sequences (brightness 120-200) as crowded.\n', '    Key changes vs v9:\n', "      - crowd > 0.65  (was 0.55)  — needs more objects before 'crowded'\n", '      - edge_density > 0.13 (was 0.10) — less sensitive to texture\n', '      - SCI crowd weight 0.35→0.30, edge weight 0.25→0.20,\n', '        tiny weight 0.25→0.30 (tiny objects matter more for UAV)\n', '    """\n', '    def __init__(self, window: int = 7):\n', '        self.sci_hist = deque(maxlen=window)\n', '\n', '    def analyze(self, img: np.ndarray, prev_boxes: np.ndarray) -> SceneState:\n', '        small = cv2.resize(img, (0, 0), fx=0.25, fy=0.25)\n', '        gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)\n', '\n', '        brightness  = float(gray.mean())\n', '        blur        = float(cv2.Laplacian(gray, cv2.CV_64F).var())\n', '        edge_density= float(cv2.Canny(gray, 50, 120).mean() / 255.0)\n', '        n_dets      = len(prev_boxes)\n', '        crowd       = min(n_dets / 30.0, 1.0)          # v10: divisor 25→30\n', '\n', '        if n_dets:\n', '            areas      = ((prev_boxes[:, 2] - prev_boxes[:, 0]) *\n', '                          (prev_boxes[:, 3] - prev_boxes[:, 1]))\n', '            tiny_ratio = float(np.mean(areas < 32 * 32))\n', '        else:\n', '            tiny_ratio = 0.0\n', '\n', '        # v10: rebalanced weights\n', '        raw_sci = (0.30 * crowd\n', '                 + 0.20 * min(edge_density / 0.14, 1.0)\n', '                 + 0.30 * tiny_ratio)\n', '        if brightness < 80:\n', '            raw_sci += 0.10\n', '        if blur < 180:\n', '            raw_sci += 0.05\n', '\n', '        self.sci_hist.append(float(np.clip(raw_sci, 0.0, 1.0)))\n', '        sci = float(np.mean(self.sci_hist))\n', '\n', '        # v10: tighter scene thresholds\n', '        if brightness < 80:\n', "            scene = 'night'\n", '        elif blur < 180:\n', "            scene = 'blur'\n", '        elif tiny_ratio > 0.50:\n', "            scene = 'tiny'\n", '        elif crowd > 0.65 or edge_density > 0.13:   # v9 was 0.55 / 0.10\n', "            scene = 'crowded'\n", '        else:\n', "            scene = 'clear'\n", '\n', '        return SceneState(sci=sci, scene=scene, brightness=brightness,\n', '                          blur=blur, edge_density=edge_density,\n', '                          crowd=crowd, tiny_ratio=tiny_ratio, n_dets=n_dets)\n', '\n', '    def reset(self):\n', '        self.sci_hist.clear()\n', '\n', '\n', 'class SmartCalibrator:\n', '    """\n', '    v10 fix:\n', '      - conf floor raised 0.17→0.19 (prevents FP explosion on UAV small objects)\n', '      - conf ceiling kept 0.28 (safe for VisDrone)\n', '      - imgsz thresholds unchanged (640/736/832 ladder)\n', '    """\n', '    def __init__(self, adaptive_threshold: bool = True,\n', '                       adaptive_resolution: bool = True):\n', '        self.adaptive_threshold  = adaptive_threshold\n', '        self.adaptive_resolution = adaptive_resolution\n', '\n', '    def params(self, state: SceneState) -> dict:\n', '        conf  = 0.25\n', '        iou   = 0.45\n', '        imgsz = 640\n', '\n', '        if self.adaptive_threshold:\n', '            conf = 0.245 - 0.050 * state.sci          # v10: slope 0.055→0.050 (gentler)\n', '            iou  = 0.490 - 0.050 * state.sci\n', "            if state.scene in ['crowded', 'tiny', 'night']:\n", '                conf -= 0.012                          # v10: nudge 0.015→0.012\n', "            if state.scene == 'blur':\n", '                iou -= 0.012\n', '\n', '        if self.adaptive_resolution:\n', '            if state.sci > 0.60 or state.tiny_ratio > 0.50:\n', '                imgsz = 832\n', "            elif state.sci > 0.35 or state.scene in ['crowded', 'tiny']:\n", '                imgsz = 736\n', '\n', '        return dict(\n', '            conf  = float(np.clip(conf,  0.19, 0.28)),   # v10: floor 0.17→0.19\n', '            iou   = float(np.clip(iou,   0.40, 0.52)),\n', '            imgsz = int(imgsz)\n', '        )\n', '\n', '\n', 'class LightweightReID:\n', '    """\n', '    Kept for ablation only — NOT used in production system in v10.\n', '    v10 ablation proved ReID increases IDS on VisDrone (A3: 157 vs A2: 128).\n', '    """\n', '    def __init__(self, crop=24, bank=4, threshold=0.85, max_age=30):\n', '        self.crop       = crop\n', '        self.bank_size  = bank\n', '        self.threshold  = threshold          # v10: raised 0.82→0.85 (stricter matching)\n', '        self.max_age    = max_age            # v10: reduced 35→30 (shorter memory)\n', '        self.bank       = defaultdict(lambda: deque(maxlen=bank))\n', '        self.lost_feat  = {}\n', '        self.lost_age   = {}\n', '        self.seen_ids   = set()\n', '        self.frame_idx  = 0\n', '\n', '    def _feature(self, img, box):\n', '        x1, y1, x2, y2 = [int(max(0, v)) for v in box]\n', '        crop = img[y1:y2, x1:x2]\n', '        if crop.size == 0:\n', '            return None\n', '        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) if crop.ndim == 3 else crop\n', '        feat = cv2.resize(gray, (self.crop, self.crop)).ravel().astype(np.float32)\n', '        feat -= feat.mean()\n', '        norm = np.linalg.norm(feat)\n', '        return feat / norm if norm > 1e-6 else None\n', '\n', '    def update_and_remap(self, img, ids, boxes):\n', '        self.frame_idx += 1\n', '        current  = set(ids.tolist()) if len(ids) else set()\n', '        remapped = ids.copy()\n', '\n', '        for i, tid in enumerate(ids):\n', '            tid  = int(tid)\n', '            feat = self._feature(img, boxes[i])\n', '            if feat is None:\n', '                continue\n', '            if tid not in self.seen_ids and self.lost_feat:\n', '                best_tid, best_sim = tid, self.threshold\n', '                for old_tid, old_feat in list(self.lost_feat.items()):\n', '                    sim = float(np.dot(feat, old_feat))\n', '                    if sim > best_sim:\n', '                        best_tid, best_sim = old_tid, sim\n', '                if best_tid != tid:\n', '                    remapped[i] = best_tid\n', '                    self.lost_feat.pop(best_tid, None)\n', '                    self.lost_age.pop(best_tid, None)\n', '                    tid = best_tid\n', '            self.bank[tid].append(feat)\n', '            self.seen_ids.add(tid)\n', '\n', '        for tid in list(self.seen_ids):\n', '            if tid not in current and tid not in self.lost_feat and self.bank[tid]:\n', '                mean_feat = np.mean(np.stack(self.bank[tid]), axis=0)\n', '                norm = np.linalg.norm(mean_feat)\n', '                if norm > 1e-6:\n', '                    self.lost_feat[tid] = mean_feat / norm\n', '                    self.lost_age[tid]  = self.frame_idx\n', '\n', '        for tid, age in list(self.lost_age.items()):\n', '            if self.frame_idx - age > self.max_age:\n', '                self.lost_feat.pop(tid, None)\n', '                self.lost_age.pop(tid, None)\n', '        return remapped\n', '\n', '\n', '# ── Helper functions ────────────────────────────────────────────\n', '\n', 'def load_gt(path: Path) -> pd.DataFrame:\n', '    if not path.exists():\n', '        return pd.DataFrame()\n', "    cols = ['frame','id','x','y','w','h','score','cat','trunc','occ']\n", '    df   = pd.read_csv(path, header=None, names=cols)\n', "    df   = df[df['cat'].isin([1,4,5,6,9])]\n", "    df   = df[(df['occ'] < 2) & (df['trunc'] < 2) & (df['score'] == 1)]\n", '    return df.reset_index(drop=True)\n', '\n', '\n', 'def iou_dist(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:\n', '    if not len(pred) or not len(gt):\n', '        return np.empty((len(gt), len(pred)))\n', '    ix1   = np.maximum(pred[:, 0:1].T, gt[:, 0:1])\n', '    iy1   = np.maximum(pred[:, 1:2].T, gt[:, 1:2])\n', '    ix2   = np.minimum(pred[:, 2:3].T, gt[:, 2:3])\n', '    iy2   = np.minimum(pred[:, 3:4].T, gt[:, 3:4])\n', '    inter = np.maximum(0, ix2 - ix1) * np.maximum(0, iy2 - iy1)\n', '    ap    = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])\n', '    ag    = (gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])\n', '    union = ap[np.newaxis, :] + ag[:, np.newaxis] - inter\n', '    return 1.0 - np.where(union > 0, inter / union, 0.0)\n', '\n', '\n', 'def hota_approx(tp: int, fp: int, fn: int, ids: int) -> float:\n', '    det_a = tp / max(tp + fp + fn, 1)\n', '    ass_a = max(0.0, 1.0 - ids / max(tp, 1))\n', '    return float(np.sqrt(det_a * ass_a))\n', '\n', '\n', "def eval_acc(acc, name='seq') -> dict:\n", '    mh   = mm.metrics.create()\n', '    summ = mh.compute(acc,\n', "        metrics=['mota','idf1','num_switches','recall','precision',\n", "                 'num_misses','num_false_positives','num_matches'],\n", '        name=name)\n', '    row  = summ.iloc[0]\n', '    return dict(\n', "        mota      = float(row['mota']),\n", "        idf1      = float(row['idf1']),\n", "        recall    = float(row['recall']),\n", "        precision = float(row['precision']),\n", "        ids       = int(row['num_switches']),\n", "        fn        = int(row['num_misses']),\n", "        fp        = int(row['num_false_positives']),\n", "        matches   = int(row['num_matches']),\n", '        hota      = hota_approx(\n', "            int(row['num_matches']), int(row['num_false_positives']),\n", "            int(row['num_misses']),  int(row['num_switches'])),\n", '    )\n', '\n', '\n', 'def build_tracker_yaml(name, high, low, new, buffer, match) -> str:\n', "    path = Path(f'/content/{name}.yaml')\n", "    data = dict(tracker_type='bytetrack',\n", '                track_high_thresh=float(high),\n', '                track_low_thresh=float(low),\n', '                new_track_thresh=float(new),\n', '                track_buffer=int(buffer),\n', '                match_thresh=float(match),\n', '                fuse_score=True)\n', "    path.write_text(yaml.safe_dump(data, sort_keys=False), encoding='utf-8')\n", '    return str(path)\n', '\n', '\n', 'def reset_tracker(model):\n', "    if getattr(model, 'predictor', None) is not None:\n", '        model.predictor = None\n', '\n', '\n', "print('AC-MOT v10 modules ready')\n", '"""Candidate A3 controller. Values are experimental, not optimized results."""\n', 'class StableCalibrator:\n', '    def __init__(self, adaptive_threshold=True, adaptive_resolution=True):\n', '        self.adaptive_threshold = adaptive_threshold\n', '        self.adaptive_resolution = adaptive_resolution\n', '        self.current = 640\n', '        self.pending = None\n', '        self.count = 0\n', '        self.last_state = None\n', '\n', '    def params(self, state):\n', '        target = 640\n', '        if self.adaptive_resolution:\n', '            if state.sci > .60 or state.tiny_ratio > .50:\n', '                target = 832\n', "            elif state.sci > .35 or state.scene in ['crowded', 'tiny']:\n", '                target = 736\n', '        # Count distinct analyzer updates, not repeated calls on the same state.\n', '        if state is not self.last_state:\n', '            self.count = self.count + 1 if target == self.pending else 1\n', '            self.pending = target\n', '            self.last_state = state\n', '            if self.count >= 3:\n', '                self.current = target\n', "        # Allow ByteTrack's low-confidence recovery stage to see detections.\n", '        # Track birth remains controlled by new_track_thresh in tracker YAML.\n', '        conf = .04 if self.adaptive_threshold else .25\n', '        iou = max(.40, min(.52, .490 - .050 * state.sci)) if self.adaptive_threshold else .45\n', '        return dict(conf=conf, iou=iou, imgsz=self.current)\n']}, {'source': ['# ════════════════════════════════════════════════════════════════\n', '#  CELL 3 — RUNNER + SYSTEMS DEFINITION\n', '# ════════════════════════════════════════════════════════════════\n', '\n', 'TRACKERS = {\n', '    # Official default ByteTrack — no tuning\n', "    'baseline': 'bytetrack.yaml',\n", '    # Tuned ByteTrack — used by both Baseline_TunedTracker AND AC-MOT_v10\n', '    # high=0.18 : ByteTrack 2nd-round uses lower-conf detections → better recall\n', '    # buffer=45 : longer track memory → fewer ID resets\n', '    # match=0.86: stricter IoU matching → fewer wrong associations\n', '    # new=0.20  : v10 fix, reduces spurious new tracks vs v9 (was 0.18)\n', "    'acmot': build_tracker_yaml('bytetrack_v10_acmot',\n", '                                high=0.18, low=0.04,\n', '                                new=0.20,  buffer=45, match=0.86),\n', '}\n', '\n', '# ── 3 production systems ─────────────────────────────────────────\n', '# System 1 vs System 2 → isolates: tracker YAML tuning contribution\n', '# System 2 vs System 3 → isolates: adaptive scene intelligence contribution\n', 'SYSTEMS = [\n', "    dict(name='Baseline_Default',\n", "         model=MODEL_NAME, tracker='baseline',\n", '         adaptive_threshold=False, adaptive_resolution=False,\n', '         scene_analysis=False, reid=False),\n', '\n', "    dict(name='Baseline_TunedTracker',\n", "         model=MODEL_NAME, tracker='acmot',       # same tuned yaml as AC-MOT\n", '         adaptive_threshold=False, adaptive_resolution=False,\n', '         scene_analysis=False, reid=False),        # zero adaptive logic\n', '\n', "    dict(name='AC-MOT_v10',\n", "         model=MODEL_NAME, tracker='acmot',\n", '         adaptive_threshold=True, adaptive_resolution=True,\n', '         scene_analysis=True, reid=False),         # ReID OFF — v9 ablation proved it increases IDS\n', ']\n', '\n', '# ── Ablation systems (4 sequences) ──────────────────────────────\n', '# A0 → A1: What does tuned YAML alone give?\n', '# A1 → A2: What does adaptive threshold add on top of tuned YAML?\n', '# A2 → A3: What does adaptive resolution add?\n', '# A3 → A4: What does ReID do? (kept as negative reference)\n', 'ABLATION_SYSTEMS = [\n', "    dict(name='A0_Baseline_Default',\n", "         model=MODEL_NAME, tracker='baseline',\n", '         adaptive_threshold=False, adaptive_resolution=False,\n', '         scene_analysis=False, reid=False),\n', '\n', "    dict(name='A1_TunedTracker_Only',\n", "         model=MODEL_NAME, tracker='acmot',\n", '         adaptive_threshold=False, adaptive_resolution=False,\n', '         scene_analysis=False, reid=False),\n', '\n', "    dict(name='A2_TunedTracker_AdaptThresh',\n", "         model=MODEL_NAME, tracker='acmot',\n", '         adaptive_threshold=True,  adaptive_resolution=False,\n', '         scene_analysis=True,  reid=False),\n', '\n', "    dict(name='A3_TunedTracker_AdaptThresh_Res',   # = production config\n", "         model=MODEL_NAME, tracker='acmot',\n", '         adaptive_threshold=True,  adaptive_resolution=True,\n', '         scene_analysis=True,  reid=False),\n', '\n', "    dict(name='A4_Full_WithReID',                   # negative reference\n", "         model=MODEL_NAME, tracker='acmot',\n", '         adaptive_threshold=True,  adaptive_resolution=True,\n', '         scene_analysis=True,  reid=True),\n', ']\n', '\n', '\n', 'def get_params(system: dict, state: SceneState) -> dict:\n', "    if system.get('adaptive_threshold') or system.get('adaptive_resolution'):\n", '        return SmartCalibrator(\n', "            adaptive_threshold  = system.get('adaptive_threshold', False),\n", "            adaptive_resolution = system.get('adaptive_resolution', False)\n", '        ).params(state)\n', '    return dict(conf=0.25, iou=0.45, imgsz=640)\n', '\n', '\n', 'def run_system(system: dict, seqs, run_tag: str) -> pd.DataFrame:\n', "    model = YOLO(system['model'])\n", '    if HALF:\n', '        model.model.model.half()\n', '\n', '    rows = []\n', "    for seq in tqdm(seqs, desc=system['name']):\n", "        gt         = load_gt(ANNOT_DIR / f'{seq.name}.txt')\n", "        frames_drv = sorted(seq.glob('*.jpg'))\n", '        if gt.empty or not frames_drv:\n', '            continue\n', '\n', '        LOCAL_TMP.mkdir(exist_ok=True)\n', '        local_seq = LOCAL_TMP / seq.name\n', '        if local_seq.exists():\n', '            shutil.rmtree(local_seq)\n', '        shutil.copytree(seq, local_seq)\n', "        frames = sorted(local_seq.glob('*.jpg'))\n", '\n', '        reset_tracker(model)\n', "        Calibrator = StableCalibrator if system.get('candidate_v11') else SmartCalibrator\n", '        calibrator   = Calibrator(\n', "                            system.get('adaptive_threshold', False),\n", "                            system.get('adaptive_resolution', False))\n", '        analyzer     = SceneAnalyzer()\n', "        reid         = LightweightReID() if system.get('reid', False) else None\n", '        acc          = mm.MOTAccumulator(auto_id=True)\n', '        times        = []\n', '        prev_boxes   = np.empty((0, 4))\n', '        state        = SceneState()\n', '        scene_counts = Counter()\n', '        imgsz_log, conf_log = [], []\n', '\n', '        for idx, fp in enumerate(frames, start=1):\n', '            t0  = time.perf_counter()\n', '            img = cv2.imread(str(fp))\n', '            if img is None:\n', '                continue\n', '\n', "            if system.get('scene_analysis') and (idx == 1 or idx % 10 == 1):\n", '                state = analyzer.analyze(img, prev_boxes)\n', '\n', '            scene_counts[state.scene] += 1\n', '            params = calibrator.params(state)\n', "            imgsz_log.append(params['imgsz'])\n", "            conf_log.append(params['conf'])\n", '\n', '            res = model.track(\n', '                source  = img,\n', "                tracker = TRACKERS[system['tracker']],\n", "                conf    = params['conf'],\n", "                iou     = params['iou'],\n", "                imgsz   = params['imgsz'],\n", '                # quantize omitted\n', '                persist = True,\n', '                verbose = False,\n', '                device  = DEVICE,\n', '            )\n', '            times.append(time.perf_counter() - t0)\n', '\n', '            if res[0].boxes.id is not None:\n', '                pred_ids   = res[0].boxes.id.cpu().numpy().astype(int)\n', '                pred_boxes = res[0].boxes.xyxy.cpu().numpy()\n', '            else:\n', '                pred_ids   = np.array([], dtype=int)\n', '                pred_boxes = np.empty((0, 4))\n', '            prev_boxes = pred_boxes.copy()\n', '\n', '            if reid is not None and len(pred_ids):\n', '                pred_ids = reid.update_and_remap(img, pred_ids, pred_boxes)\n', '\n', "            gt_f     = gt[gt['frame'] == idx]\n", "            gt_ids   = gt_f['id'].values\n", '            gt_boxes = (np.column_stack([\n', "                gt_f['x'].values, gt_f['y'].values,\n", "                gt_f['x'].values + gt_f['w'].values,\n", "                gt_f['y'].values + gt_f['h'].values,\n", '            ]) if len(gt_f) else np.empty((0, 4)))\n', '\n', '            dist = iou_dist(pred_boxes, gt_boxes)\n', '            acc.update(gt_ids, pred_ids,\n', '                       dist if dist.size else np.empty((len(gt_ids), len(pred_ids))))\n', '\n', '        shutil.rmtree(local_seq, ignore_errors=True)\n', '        metrics = eval_acc(acc, seq.name)\n', '        fps     = 1.0 / np.mean(times) if times else 0.0\n', "        dom     = scene_counts.most_common(1)[0][0] if scene_counts else 'unknown'\n", '\n', '        rows.append(dict(\n', '            run_tag        = run_tag,\n', "            system         = system['name'],\n", "            tracker_cfg    = system['tracker'],       # logged for reproducibility\n", '            sequence       = seq.name,\n', '            frames         = len(frames),\n', '            fps            = round(fps, 2),\n', '            dominant_scene = dom,\n', '            mean_imgsz     = round(float(np.mean(imgsz_log)), 1) if imgsz_log else 640,\n', '            mean_conf      = round(float(np.mean(conf_log)),  4) if conf_log  else 0.25,\n', '            **metrics,\n', '        ))\n', '        tqdm.write(\n', '            f"{system[\'name\']:<34} {seq.name[:24]:24s} "\n', '            f"MOTA={metrics[\'mota\']:.3f} IDF1={metrics[\'idf1\']:.3f} "\n', '            f"HOTA={metrics[\'hota\']:.3f} IDS={metrics[\'ids\']:4d} "\n', '            f"FPS={fps:.1f} scene={dom}"\n', '        )\n', '\n', '    if HALF:\n', '        torch.cuda.empty_cache()\n', '    gc.collect()\n', '    return pd.DataFrame(rows)\n', '\n', '\n', 'def summarize(df: pd.DataFrame) -> pd.DataFrame:\n', '    rows = []\n', "    for system, g in df.groupby('system', sort=False):\n", '        rows.append(dict(\n', '            system     = system,\n', '            sequences  = len(g),\n', "            mota       = g['mota'].mean(),\n", "            idf1       = g['idf1'].mean(),\n", "            recall     = g['recall'].mean(),\n", "            precision  = g['precision'].mean(),\n", "            hota       = g['hota'].mean(),\n", "            ids        = int(g['ids'].sum()),\n", "            fn         = int(g['fn'].sum()),\n", "            fp         = int(g['fp'].sum()),\n", "            matches    = int(g['matches'].sum()),\n", "            fps        = g['fps'].mean(),\n", "            mean_imgsz = g['mean_imgsz'].mean(),\n", '        ))\n', '    out = pd.DataFrame(rows)\n', '    if len(out) >= 2:\n', '        base = out.iloc[0]\n', "        for col in ['mota','idf1','recall','precision','hota','fps']:\n", "            out[col + '_delta'] = out[col] - float(base[col])\n", "        out['ids_delta'] = out['ids'] - int(base['ids'])\n", "        out['fn_delta']  = out['fn']  - int(base['fn'])\n", "        out['fp_delta']  = out['fp']  - int(base['fp'])\n", '    return out\n', '\n', '\n', "print('Runner ready')\n", "print('Systems: Baseline_Default | Baseline_TunedTracker | AC-MOT_v10')\n", "print('Tracker configs logged per-row for reproducibility')\n", "ABLATION_SYSTEMS[3]['candidate_v11'] = True\n", "ABLATION_SYSTEMS[3]['name'] = 'A3_v11_Candidate'\n"]}]

In [ ]:
%%writefile /content/record_full17.py
"""Run with %run -i after the FULL17 notebook setup/main run.

Records all five ablations, per-frame tracking, detector outputs and settings.
Existing v10 evaluator is retained and explicitly labelled as legacy/proxy.
Replay videos are rendered only from recorded tracking; rendering needs no YOLO.
"""
import gzip
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import traceback
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(value, indent=2, default=str))
    temp.replace(path)


assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name(0)
assert len(VAL_SEQS) == 17
RECORD_ROOT = Path(globals().get("RECORDED_RUN_DIR", DRIVE_RESULTS / (
    "acmot_full17_recorded_" + datetime.now().strftime("%Y%m%d_%H%M%S"))))
RECORD_ROOT.mkdir(parents=True, exist_ok=True)
RECORDED_RUN_DIR = str(RECORD_ROOT)
print("RECORD_ROOT:", RECORD_ROOT, flush=True)

# Freeze implementation and configuration before any additional inference.
original_sources = ["".join(c["source"]) for c in code_cells[:3]]
for i, source in enumerate(original_sources, 1):
    target = RECORD_ROOT / f"original_cell_{i}.py"
    if not target.exists():
        target.write_text(source)
suite_source = Path(__file__).read_text() if "__file__" in globals() else ""
if suite_source:
    (RECORD_ROOT / "record_full17.py").write_text(suite_source)
frozen_systems = [dict(s) for s in ABLATION_SYSTEMS]
assert len(frozen_systems) == 5
from ultralytics.utils.checks import check_yaml
frozen_trackers = {k: Path(check_yaml(v)).read_text() for k, v in TRACKERS.items()}
fingerprint = hashlib.sha256(json.dumps([original_sources, frozen_systems,
    frozen_trackers, suite_source], sort_keys=True).encode()).hexdigest()
configuration = dict(fingerprint=fingerprint, gpu=torch.cuda.get_device_name(0),
    dataset=str(SEQ_DIR.parent), python=platform.python_version(),
    packages={p: importlib.metadata.version(p) for p in
              ["torch", "ultralytics", "motmetrics", "numpy", "pandas", "opencv-python-headless"]},
    systems=frozen_systems, tracker_yamls=frozen_trackers,
    ground_truth_filter=dict(categories=[1,4,5,6,9], score=1, occlusion_lt=2, truncation_lt=2),
    metric_protocol="Original v10: unthresholded IoU assignment; HOTA is a proxy, not official HOTA",
    fps_protocol="GPU synchronized inference/tracking including image read and scene analysis; excludes record serialization",
    replay_scope="Exact recorded predictions and visual replay; detector/model changes require new inference")
config_path = RECORD_ROOT / "configuration.json"
if config_path.exists():
    assert json.loads(config_path.read_text())["fingerprint"] == fingerprint, "Resume configuration differs"
else:
    atomic_json(config_path, configuration)
subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=open(RECORD_ROOT / "requirements-lock.txt", "w"), check=True)
atomic_json(RECORD_ROOT / "status.json", dict(status="validating_dataset"))

manifest = []
for seq in VAL_SEQS:
    frames = sorted(seq.glob("*.jpg"))
    ids = [int(p.stem) for p in frames]
    assert ids == list(range(1, len(frames)+1)), f"Missing frames: {seq.name}"
    ann = ANNOT_DIR / f"{seq.name}.txt"
    assert ann.is_file() and not load_gt(ann).empty, seq.name
    raw_gt = pd.read_csv(ann, header=None)
    assert int(raw_gt.iloc[:,0].max()) <= len(frames), seq.name
    file_hashes = {p.name: sha256(p) for p in frames}
    manifest.append(dict(sequence=seq.name, frames=len(frames), annotation_sha256=sha256(ann),
        annotation_rows=len(raw_gt), filtered_gt_rows=len(load_gt(ann)), frame_sha256=file_hashes))
atomic_json(RECORD_ROOT / "dataset_manifest.json", manifest)

# Instrument the unchanged system logic. One gzip JSONL record per input frame.
_detector_output = []
def capture_detections(predictor):
    global _detector_output
    b = predictor.results[0].boxes
    _detector_output = b.data.detach().cpu().numpy().tolist() if b is not None else []

def record_frame(model, seq, index, frame_path, pred_ids, pred_boxes, state, params, elapsed, result):
    b = result.boxes
    payload = dict(frame=index, filename=frame_path.name,
        ids=pred_ids.tolist(), boxes_xyxy=pred_boxes.tolist(),
        scores=b.conf.detach().cpu().numpy().tolist() if b is not None else [],
        classes=b.cls.detach().cpu().numpy().astype(int).tolist() if b is not None else [],
        detections_before_tracking=_detector_output,
        scene=dict(vars(state)), settings=dict(params, device=DEVICE, half_requested=HALF),
        elapsed_seconds=elapsed,
        actual_model_fp16=bool(getattr(getattr(model.predictor, "model", None), "fp16", False)))
    _record_stream.write(json.dumps(payload, separators=(",", ":")) + "\n")

instrumented = original_sources[2]
assert instrumented.count("    model = YOLO(system['model'])") == 1
instrumented = instrumented.replace("    model = YOLO(system['model'])",
    "    model = YOLO(system['model'])\n    model.add_callback('on_predict_postprocess_end', capture_detections)")
instrumented = instrumented.replace("            t0  = time.perf_counter()",
    "            torch.cuda.synchronize()\n            t0  = time.perf_counter()")
instrumented = instrumented.replace("            times.append(time.perf_counter() - t0)",
    "            torch.cuda.synchronize()\n            times.append(time.perf_counter() - t0)")
needle = "            gt_f     = gt[gt['frame'] == idx]"
assert instrumented.count(needle) == 1
instrumented = instrumented.replace(needle,
    "            record_frame(model, seq, idx, fp, pred_ids, pred_boxes, state, params, times[-1], res[0])\n" + needle)
(RECORD_ROOT / "instrumented_runner.py").write_text(instrumented)
exec(compile(instrumented, str(RECORD_ROOT / "instrumented_runner.py"), "exec"), globals())

completed_rows = []
for system in frozen_systems:
    sysdir = RECORD_ROOT / system["name"]
    sysdir.mkdir(exist_ok=True)
    for seq in VAL_SEQS:
        row_path = sysdir / f"{seq.name}.metrics.json"
        record_path = sysdir / f"{seq.name}.frames.jsonl.gz"
        if row_path.exists() and record_path.exists():
            saved = json.loads(row_path.read_text())
            assert saved["fingerprint"] == fingerprint
            assert saved["record_sha256"] == sha256(record_path)
            completed_rows.append(saved["metrics"])
            continue
        atomic_json(RECORD_ROOT / "status.json", dict(status="running", system=system["name"],
            sequence=seq.name, completed=len(completed_rows), total=85))
        part = sysdir / f"{seq.name}.frames.partial.jsonl.gz"
        try:
            with gzip.open(part, "wt", encoding="utf-8") as _record_stream:
                result_df = run_system(system, [seq], RECORD_ROOT.name)
            assert len(result_df) == 1, f"Missing result: {seq.name}"
            with gzip.open(part, "rt") as f:
                saved_frames = [json.loads(line)["frame"] for line in f]
            count = len(list(seq.glob("*.jpg")))
            assert saved_frames == list(range(1, count+1)), f"Incomplete recordings: {seq.name}"
            part.replace(record_path)
            metric = result_df.iloc[0].to_dict()
            atomic_json(row_path, dict(fingerprint=fingerprint, record_sha256=sha256(record_path), metrics=metric))
            completed_rows.append(metric)
            pd.DataFrame(completed_rows).to_csv(RECORD_ROOT / "per_sequence_checkpoint.csv", index=False)
            print(f"RECORDED {len(completed_rows)}/85: {system['name']} / {seq.name}", flush=True)
        except Exception:
            atomic_json(RECORD_ROOT / "status.json", dict(status="failed", system=system["name"],
                sequence=seq.name, error=traceback.format_exc(), completed=len(completed_rows)))
            raise
        weight = Path(system["model"])
        if weight.is_file() and not (RECORD_ROOT / weight.name).exists():
            shutil.copy2(weight, RECORD_ROOT / weight.name)
            atomic_json(RECORD_ROOT / "weights.json", dict(filename=weight.name, sha256=sha256(weight)))

recorded_df = pd.DataFrame(completed_rows)
assert len(recorded_df) == 85
assert all(len(g) == 17 and set(g.sequence) == set(VAL_SEQS_NAMES) for _, g in recorded_df.groupby("system"))
recorded_summary = summarize(recorded_df)
recorded_df.to_csv(RECORD_ROOT / "per_sequence.csv", index=False)
recorded_summary.to_csv(RECORD_ROOT / "summary.csv", index=False)
atomic_json(RECORD_ROOT / "status.json", dict(status="recording_complete", completed=85, total=85))
print("RECORDING COMPLETE", RECORD_ROOT, flush=True)
print(recorded_summary.to_string(index=False))

# Standalone replay program is copied next to the records by the notebook.
replay_script = Path("/content/render_recorded.py")
if replay_script.exists():
    shutil.copy2(replay_script, RECORD_ROOT / "render_recorded.py")
    subprocess.run([sys.executable, str(replay_script), str(RECORD_ROOT)], check=True)
else:
    print("Recordings saved. Run render_recorded.py with RECORD_ROOT to produce videos.")


In [ ]:
%%writefile /content/evaluate_v11.py
"""Official TrackEval metrics on preserved, class-agnostic research-filter tracks.

This adapter is NOT the official VisDrone benchmark preprocessing protocol.
"""
import argparse
import gzip
import hashlib
import json
import subprocess
import sys
from pathlib import Path
import numpy as np

REVISION = '12c8791b303e0a0b50f753af204249e622d0281a'

# Pinned upstream uses removed NumPy scalar aliases. Restore aliases only;
# no metric equations or matching logic are changed.
for alias, scalar in [('float', float), ('int', int)]:
    if alias not in np.__dict__:
        setattr(np, alias, scalar)

def iou(a, b):
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    b = np.asarray(b, dtype=float).reshape(-1, 4)
    for boxes in (a, b):
        if not np.isfinite(boxes).all() or np.any(boxes[:, 2:] <= boxes[:, :2]):
            raise ValueError('Non-finite or non-positive box')
    inter = np.maximum(0, np.minimum(a[:, None, 2:], b[None, :, 2:]) -
                       np.maximum(a[:, None, :2], b[None, :, :2])).prod(axis=2)
    union = (a[:, 2:] - a[:, :2]).prod(axis=1)[:, None] + (b[:, 2:] - b[:, :2]).prod(axis=1)[None, :] - inter
    return np.divide(inter, union, out=np.zeros_like(inter), where=union > 0)

def prepare(gt, frames, count):
    if [r['frame'] for r in frames] != list(range(1, count + 1)):
        raise ValueError('Recording must contain every frame exactly once, in order')
    gt = np.asarray(gt, dtype=float).reshape(-1, 10)
    if not np.isfinite(gt).all() or np.any(gt[:, 0] < 1) or np.any(gt[:, 0] > count):
        raise ValueError('Invalid annotation frame or value')
    gt = gt[np.isin(gt[:, 7], [1, 4, 5, 6, 9]) & (gt[:, 6] == 1) & (gt[:, 8] < 2) & (gt[:, 9] < 2)]
    gids = sorted(set(gt[:, 1].tolist()))
    pids = sorted({v for r in frames for v in r['ids']})
    gm, pm = {v:i for i,v in enumerate(gids)}, {v:i for i,v in enumerate(pids)}
    data = dict(num_timesteps=count, num_gt_ids=len(gids), num_tracker_ids=len(pids),
                num_gt_dets=len(gt), num_tracker_dets=sum(len(r['ids']) for r in frames),
                gt_ids=[], tracker_ids=[], similarity_scores=[])
    for r in frames:
        g = gt[gt[:, 0] == r['frame']]
        ids = r['ids']
        if len(ids) != len(r['boxes_xyxy']) or len(set(ids)) != len(ids) or len(set(g[:, 1])) != len(g):
            raise ValueError('Duplicate IDs or mismatched boxes/IDs')
        gb = g[:, 2:6].copy()
        gb[:, 2:] += gb[:, :2]
        data['gt_ids'].append(np.array([gm[v] for v in g[:, 1]], dtype=int))
        data['tracker_ids'].append(np.array([pm[v] for v in ids], dtype=int))
        data['similarity_scores'].append(iou(gb, r['boxes_xyxy']))
    return data

def main():
    p = argparse.ArgumentParser()
    p.add_argument('run', type=Path)
    p.add_argument('--dataset', required=True, type=Path)
    p.add_argument('--trackeval', required=True, type=Path)
    p.add_argument('--output', required=True, type=Path)
    a = p.parse_args()
    rev = subprocess.check_output(['git', '-C', str(a.trackeval), 'rev-parse', 'HEAD'], text=True).strip()
    if rev != REVISION:
        raise ValueError('TrackEval revision differs from pinned revision')
    sys.path.insert(0, str(a.trackeval))
    import trackeval
    metrics = [trackeval.metrics.HOTA(), trackeval.metrics.CLEAR({'THRESHOLD': .5, 'PRINT_CONFIG': False}),
               trackeval.metrics.Identity({'THRESHOLD': .5, 'PRINT_CONFIG': False})]
    cfg = json.loads((a.run/'configuration.json').read_text())
    manifest = json.loads((a.run/'dataset_manifest.json').read_text())
    results, hashes = {}, {}
    for system in cfg['systems']:
        name = system['name']
        per_metric = {m.get_name(): {} for m in metrics}
        for seq in manifest:
            sn = seq['sequence']
            ann = a.dataset/'annotations'/f'{sn}.txt'
            digest = hashlib.sha256(ann.read_bytes()).hexdigest()
            if digest != seq['annotation_sha256']:
                raise ValueError(f'Annotation hash differs: {sn}')
            recording = a.run/name/f'{sn}.frames.jsonl.gz'
            hashes[str(recording)] = hashlib.sha256(recording.read_bytes()).hexdigest()
            with gzip.open(recording, 'rt') as f:
                frames = [json.loads(line) for line in f]
            data = prepare(np.loadtxt(ann, delimiter=',', ndmin=2), frames, seq['frames'])
            for m in metrics:
                per_metric[m.get_name()][sn] = m.eval_sequence(data)
        results[name] = {m.get_name(): dict(per_sequence=per_metric[m.get_name()],
            combined=m.combine_sequences(per_metric[m.get_name()])) for m in metrics}
    a.output.mkdir(parents=True, exist_ok=False)
    def serial(v):
        return v.tolist() if hasattr(v, 'tolist') else v
    (a.output/'metrics.json').write_text(json.dumps(results, default=serial, indent=2, allow_nan=False))
    import csv
    with (a.output/'summary.csv').open('w') as f:
        writer = csv.writer(f)
        writer.writerow(['system', 'HOTA', 'DetA', 'AssA', 'MOTA', 'IDF1', 'IDS', 'FN', 'FP'])
        for name, r in results.items():
            h, c, i = [r[k]['combined'] for k in ['HOTA', 'CLEAR', 'Identity']]
            writer.writerow([name, *[float(np.mean(h[k])) for k in ['HOTA','DetA','AssA']],
                             c['MOTA'], i['IDF1'], c['IDSW'], c['CLR_FN'], c['CLR_FP']])
    (a.output/'protocol.json').write_text(json.dumps(dict(trackeval_commit=rev,
        protocol='Custom class-agnostic research filter; no benchmark ignore-region preprocessing',
        official_visdrone=False, gt_filter=cfg['ground_truth_filter'],
        aggregation='TrackEval combine_sequences, not macro mean', scale='0–1',
        source_hashes=hashes, evaluator_sha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()), indent=2))
    print(a.output/'summary.csv')

if __name__ == '__main__':
    main()


In [ ]:
# Explicit execution cell: performs new T4 inference for all five systems.
%run -i /content/record_full17.py


In [ ]:
# Can also run separately against the OLD recorded run: no inference needed.
import subprocess, sys
from pathlib import Path
repo = Path('/content/TrackEval_acmot')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/JonathonLuiten/TrackEval.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'checkout','12c8791b303e0a0b50f753af204249e622d0281a'],check=True)
subprocess.run([sys.executable,'/content/evaluate_v11.py',RECORDED_RUN_DIR,'--dataset',str(DATASET_ROOT),'--trackeval',str(repo),'--output',str(Path(RECORDED_RUN_DIR)/'trackeval_research_v11')],check=True)
